In [ ]:
!pip install transformers

In [ ]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
from transformers import pipeline
import time
import json
import re
import pandas as pd
from collections import Counter

In [ ]:
# MODEL = "hatmimoha/arabic-ner"
MODEL = "Davlan/distilbert-base-multilingual-cased-ner-hrl"
# MODEL = "Babelscape/wikineural-multilingual-ner" # 87%
# MODEL = ""

NUM = 500
label_convert = {
    "B-LOCATION":"loc", "I-LOCATION":"loc", "I-EVENT":"evnt","B-EVENT":"evnt",
    "B-ORGANIZATION":"org","I-ORGANIZATION":"org","I-PERSON":"per", "B-PERSON":"per",
    "O":"O", "B-COMPETITION": "per", "B-DATE": "misc", "B-DISEASE":  "misc", "B-PRICE": "misc", "B-PRODUCT": "misc",
    "I-COMPETITION": "per", "I-DATE": "misc", "O":  "O", "I-PRICE": "misc", "I-PRODUCT": "misc","I-DISEASE":"misc",
    "B-LOC":"loc", "I-LOC":"loc", "B-ORG":"org","I-ORG":"org","I-PER":"per", "B-PER":"per",
    "B-MISC":"O", "I-MISC":"O", "B-PERS":"per", "I-PERS":"per",
}

In [ ]:

# Load the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL)

nlp = pipeline("ner", model=model, tokenizer=tokenizer)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/539M [00:00<?, ?B/s]

In [ ]:
def has_no_arabic_chars(word):
    pattern =  r'[«#>\]_*,;%@+/:^)~»<?$؛({\'!}=.\[\\"`،|\-&]'
    try: value = re.sub(pattern, '', word)
    except: print(word)
    return value


def clean_data(word, label):
  arabic_word = has_no_arabic_chars(word)
  if  arabic_word =='': return [], []
  else : return arabic_word, label_convert[label]



def clean_lines (line, labels):
  t_results ,l_results = [], []

  text = line.split()
  line_labels = labels.split()
  for i, _ in enumerate(text) :
    t, l = clean_data(text[i], line_labels[i])
    t_results.append(t); l_results.append(l)

  return t_results, l_results

def split_text_file(df):
    data = [];
    for i in range(NUM):
      try:
        # print(df['text'][i], df['labels'][i])
        line, labels = clean_lines(df['text'][i], df['labels'][i])
        data.append([line, labels])
      except Exception as e :print(e, flush=True)
    return data



In [ ]:


data= pd.read_csv("/content/drive/MyDrive/NLP/NER/dataset/b.txt", sep='\t', header=None)
data.columns = ["text", "labels"]
data.dropna(inplace=True)
data = data.reset_index(drop=True)
data.shape

(922, 2)

In [ ]:
data_ar = split_text_file(data[:NUM])


In [ ]:

def start_porcess(text, nlp):
  annotations = nlp(text)
  lines, entities = [], []
  for idx,  sentence in enumerate(annotations):
    if sentence == []:  lines.append(text[idx]); entities.append(label_convert["O"])
    else :
      entitie = [label_convert[temp["entity"]] for temp in sentence]
      entity_counts = Counter(entitie)
      most_repeated_entity = entity_counts.most_common(1)[0][0]
      lines.append(text[idx]); entities.append(most_repeated_entity)
  return lines, entities

def predict_data(sentences, nlp):
  lines, entities = [], []
  for idx,  sentence in enumerate(sentences):
    line, entity = start_porcess(sentence[0], nlp)
    lines.append(line); entities.append(entity)
  return lines, entities


In [ ]:
text = data_ar[:NUM]
lines, predicted_entities = predict_data(text, nlp)

print(predicted_entities)


[['O', 'org', 'per', 'loc', 'O', 'O', 'O', 'per', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'loc', 'O', 'loc', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'per'], ['loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'per', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'org', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'

In [ ]:
entities = [data[1] for data in data_ar]
print(entities)

[['loc', 'loc', 'per', 'per', 'O', 'O', 'O', 'per', 'per', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'loc', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['per', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'loc', 'loc', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'loc', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'per', 'pe

In [ ]:

grouped_data = {
    'Word':[item for sublist in lines for item in sublist],
    'Predicted': [item for sublist in predicted_entities for item in sublist],
    'Target': [item for sublist in entities for item in sublist]
    }

df = pd.DataFrame(grouped_data)


df

,Word,Predicted,Target
0,الصالحية,O,loc
1,المفرق,org,loc
2,غيث,per,per
3,الطراونة,loc,per
4,أمر,O,O
...,...,...,...
12196,العالم,O,O
12197,والتي,O,O
12198,تبلغ,O,O
12199,نحو,O,O


In [ ]:
df.Predicted.unique(), df.Target.unique()

(array(['O', 'org', 'per', 'loc'], dtype=object),
 array(['loc', 'per', 'O', 'org'], dtype=object))

In [ ]:
df = df.dropna()

In [ ]:

from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(df['Target'], df['Predicted'], labels=['loc', 'org', 'per', 'O'])


In [ ]:
conf_matrix

array([[ 366,   16,   13,   65],
       [  42,  143,   39,  183],
       [ 116,   66,  434,   92],
       [ 374,  184,  184, 9884]])

In [ ]:
from sklearn.metrics import classification_report
report = classification_report(df['Target'], df['Predicted'], labels=['loc', 'org', 'per', 'O'])
print(report)

              precision    recall  f1-score   support

         loc       0.41      0.80      0.54       460
         org       0.35      0.35      0.35       407
         per       0.65      0.61      0.63       708
           O       0.97      0.93      0.95     10626

    accuracy                           0.89     12201
   macro avg       0.59      0.67      0.62     12201
weighted avg       0.91      0.89      0.89     12201



In [ ]:
from sklearn.metrics import f1_score
labels_ = {}
f1_score(df['Target'], df['Predicted'], average='macro')


0.6168805243690555

In [ ]:
f1_score(df['Target'], df['Predicted'], average='micro')


0.8873862798131301

In [ ]:
f1_score(df['Target'], df['Predicted'], average='weighted')

0.8942824079971801

In [ ]:
f1_scores =f1_score(df['Target'], df['Predicted'], average=None)
f1_scores

array([0.53902798, 0.3504902 , 0.6298984 , 0.94810552])

In [ ]:

results = pd.DataFrame(conf_matrix, columns=['loc', 'misc', 'org', 'per', 'O'])
results['f1_score'] = f1_scores

results

,loc,misc,org,per,O,f1_score
0,507,0,93,6,82,0.485400
1,68,21,28,13,261,0.059914
2,121,1,254,19,304,0.368383
3,434,12,67,528,416,0.490023
4,271,276,238,132,18011,0.947897


import re

def has_no_arabic_chars(word):
             
    pattern =  r'[«#>\]_*,;%@+/:^)~»<?$؛({\'!}=.\[\\"`،|\-&]'
    return re.sub(pattern, '', word)


def clean_data(word, label):
  if has_no_arabic_chars(word) =='': return [], []
  else : return has_no_arabic_chars(word), label

def split_text_file(filename):
    with open(filename ,'r', encoding='utf-8') as file:
        lines = file.readlines()
    data = [];
    for line in lines:
        line = line.strip()
        if line:
            parts = line.split('\t')
            text = parts[0]
            _sentence = text.split(" ")
            _labels = parts[1:][0].split(" ")
            sentence= []; labels =[]
            for i, word in enumerate(_sentence):
              tmp_sen, tmp_leb = clean_data(word, _labels[i])
              if tmp_sen != []: sentence.append(tmp_sen); labels.append(tmp_leb)
            data.append((sentence, labels))

    return data

data= split_text_file("/content/drive/MyDrive/projects/NER/a.txt")
------------


import re

def split_text_file(filename):
    with open(filename ,'r', encoding='utf-8') as file:
        lines = file.readlines()

    data = []
    for line in lines:
        line = line.strip()
        if line:
            parts = line.split('\t')
            text = parts[0]
            labels = parts[1:]
            words = text.split()
            data.append((words, labels))


    return data

data= split_text_file("/content/drive/MyDrive/projects/NER/a.txt")
